# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mr-PeterMaged/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane 2 (Refresh/Content Opportunity Scoring)** was framed back in ML-03 as: classify
`is_declining_label`, then rank by that probability for a capacity-limited reviewer
(precision@K). Per the training-honest-models menu, a "yes/no with an observed label" task
starts with **Logistic Regression, then Random Forest** -- readable first, stronger second --
and gets evaluated the "which first?" way, at precision@K, since that's the real decision
(a reviewer only ever looks at the top of the list).

**What goes in the comparison table, and why each belongs:**
- **Dummy (stratified)** -- the floor below the floor. If nothing beats this, nothing here
  works.
- **Baseline rule (ML-07, frozen)** -- `low_ctr_visible_page`, unchanged from last week. Per
  the skill: "keep the baseline frozen once model work starts."
- **Logistic Regression** -- readable, and cheap to try. Shown twice (unscaled, then scaled)
  because the first attempt hits a real, worth-showing pitfall (section 3).
- **Random Forest** -- the complexity is only justified if it actually earns a meaningfully
  higher precision@K than the simpler options above, checked honestly in section 3, not
  assumed.

In [1]:
import sklearn, pandas, numpy
print("versions -- sklearn:", sklearn.__version__, "| pandas:", pandas.__version__, "| numpy:", numpy.__version__)
RANDOM_SEED = 42  # fixed everywhere below, named once so it's easy to audit

versions -- sklearn: 1.9.0 | pandas: 3.0.5 | numpy: 2.5.2


## 2. Split design

**Grouped by client (`GroupShuffleSplit`, 70/30, `random_state=42`).** Splitting by row would
let the same client appear in both train and test -- the model could learn client-specific
quirks instead of a generalizable pattern, and precision@K would look better than it will on a
brand-new client. Grouping by `client_hash_id` is the honest version, and it's the same design
`flyrank-data` recommends and the starter pipeline already uses.

**Same universe as the ML-07 baseline** (`avg_position_first_half` in `(0, 20]`,
`impressions_first_half >= 200`), narrowed by one more honest filter carried over from ML-04:
`clicks_first_half > 0`, because `is_declining_label` (second-half clicks < first-half clicks)
isn't a meaningful comparison starting from zero clicks. **Same decision point** as ML-04/07
(`2026-03-15`), **same excluded field** (`content_updated_date` -- postdates the decision point
for most rows, still not used).

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

pd.set_option("display.width", 160)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DECISION_DATE = pd.Timestamp("2026-03-15")

df = con.execute(f"""
    WITH base AS (SELECT * FROM '{MARCH}' WHERE gsc_data_available IS TRUE),
    first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_first_half,
               SUM(gsc_clicks) AS clicks_first_half,
               AVG(gsc_avg_position) AS avg_position_first_half,
               SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END)
                   AS ga4_engaged_sessions_first_half,
               COUNT(DISTINCT report_date) AS days_active_first_half
        FROM base
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_second_half
        FROM base
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT f.*, COALESCE(s.clicks_second_half, 0) AS clicks_second_half, dc.content_created_date
    FROM first_half f
    LEFT JOIN second_half s USING (client_hash_id, content_hash_id)
    JOIN '{DIM_CONTENT}' dc USING (client_hash_id, content_hash_id)
    WHERE f.clicks_first_half > 0
      AND dc.is_published IS TRUE AND dc.is_deleted IS FALSE
      AND dc.content_created_date <= DATE '2026-03-15'
""").df()

df["is_declining_label"] = (df["clicks_second_half"] < df["clicks_first_half"]).astype(int)
df["ctr_first_half"] = df["clicks_first_half"] / df["impressions_first_half"]
df["content_age_days"] = (DECISION_DATE - pd.to_datetime(df["content_created_date"])).dt.days

def pos_tier(p):
    if p <= 0: return "no_rank"
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"
df["position_tier"] = df["avg_position_first_half"].apply(pos_tier)

universe = df[(df["avg_position_first_half"] > 0) & (df["avg_position_first_half"] <= 20)
              & (df["impressions_first_half"] >= 200)].copy()
# DuckDB's parquet scan order isn't guaranteed run to run; sort on a stable key so the
# GroupShuffleSplit (which orders unique groups by first appearance) reproduces exactly.
universe = universe.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

print(f"universe: {len(universe):,} rows, {universe['client_hash_id'].nunique()} clients, "
      f"label positive rate {universe['is_declining_label'].mean():.1%}")

from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(universe, groups=universe["client_hash_id"]))
train, test = universe.iloc[train_idx].copy(), universe.iloc[test_idx].copy()

print(f"train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients, "
      f"label rate {train['is_declining_label'].mean():.1%}")
print(f"test:  {len(test):,} rows, {test['client_hash_id'].nunique()} clients, "
      f"label rate {test['is_declining_label'].mean():.1%}")
assert set(train["client_hash_id"]).isdisjoint(set(test["client_hash_id"])), "client leaked across split!"
print("no client appears in both train and test: confirmed")

universe: 37,026 rows, 33 clients, label positive rate 51.8%
train: 19,223 rows, 23 clients, label rate 50.0%
test:  17,803 rows, 10 clients, label rate 53.7%
no client appears in both train and test: confirmed


## 3. Train + compare vs my baseline

The ML-07 rule's peer-median CTR thresholds are **reused exactly as computed last week**
(`1-3: 0.2783%`, `4-10: 0.2101%`, `11-20: 0.1860%`) -- not recomputed on this split, per "keep
the baseline frozen." Everything is evaluated on the **same test rows, same label
(`is_declining_label`), same metric (precision@K)**.

In [3]:
FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
            "avg_position_first_half", "ga4_engaged_sessions_first_half",
            "days_active_first_half", "content_age_days"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# frozen ML-07 baseline score (peer-median CTR by position tier, from last week's notebook)
PEER_MEDIAN_CTR = {"1-3": 0.002783, "4-10": 0.002101, "11-20": 0.001860}
test = test.copy()
test["peer_median_ctr"] = test["position_tier"].map(PEER_MEDIAN_CTR)
test["baseline_score"] = (
    (test["peer_median_ctr"] - test["ctr_first_half"]).clip(lower=0) * test["impressions_first_half"]
)

X_train, y_train = train[FEATURES].fillna(0), train["is_declining_label"]
X_test, y_test = test[FEATURES].fillna(0), test["is_declining_label"]

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

scores = {}

dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_SEED).fit(X_train, y_train)
scores["Dummy (stratified)"] = dummy.predict_proba(X_test)[:, 1]

scores["Baseline rule (ML-07, frozen)"] = test["baseline_score"].values

lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)
scores["Logistic Regression (unscaled)"] = lr.predict_proba(X_test)[:, 1]

scaler = StandardScaler().fit(X_train)
lr_s = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
scores["Logistic Regression (scaled)"] = lr_s.predict_proba(scaler.transform(X_test))[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                             random_state=RANDOM_SEED, n_jobs=-1).fit(X_train, y_train)
scores["Random Forest"] = rf.predict_proba(X_test)[:, 1]

rows = []
for name, s in scores.items():
    rows.append({
        "method": name,
        "precision@20": precision_at_k(s, y_test.values, 20),
        "precision@50": precision_at_k(s, y_test.values, 50),
        "precision@100": precision_at_k(s, y_test.values, 100),
        "roc_auc": roc_auc_score(y_test, s),
    })
comparison = pd.DataFrame(rows).set_index("method")
print(f"base rate (test, K=doesn't matter): {y_test.mean():.3f}\n")
print(comparison.round(3))

base rate (test, K=doesn't matter): 0.537

                                precision@20  precision@50  precision@100  roc_auc
method                                                                            
Dummy (stratified)                      0.50          0.54           0.54    0.499
Baseline rule (ML-07, frozen)           0.60          0.54           0.50    0.436
Logistic Regression (unscaled)          0.35          0.32           0.33    0.545
Logistic Regression (scaled)            0.45          0.58           0.60    0.576
Random Forest                           0.70          0.78           0.76    0.604


**Reading the table:**

- **Baseline rule ROC-AUC = 0.436 -- below 0.5.** The CTR-fix rule was built to find pages
  under-clicking *relative to their position peers*, not pages about to lose clicks
  second-half -- it was never aimed at this label, and the comparison shows it honestly: on
  *this* metric it's slightly worse than random. That's not a bug in either notebook, it's two
  rules answering two different questions, made visible by holding both to the same test.
- **Logistic Regression (unscaled) is worse than the Dummy floor at precision@50 (0.32 vs
  0.54)** despite a better-than-random ROC-AUC (0.545) -- a real pitfall, not noise.
  `impressions_first_half` (values in the thousands-to-tens-of-thousands) dwarfs
  `ctr_first_half` (values under 0.03) in raw scale, so the unscaled fit's top-ranked rows are
  dominated by whichever feature has the largest numbers, not the most predictive one.
  **Scaling fixes it**: precision@50 goes from 0.32 to 0.58, beating both the dummy and the
  baseline. Lesson kept for next time: scale features before Logistic Regression, always.
- **Random Forest wins clearly**: precision@50 = 0.78 (vs 0.58 for scaled LR, 0.54 for both
  dummy and baseline), and it holds up at precision@20 (0.70) and precision@100 (0.76) too --
  not a fluke of one K. This is the complexity earning its place, not decoration: the gap over
  the strongest simple model (scaled LR) is 20 points at precision@50, not 2.

## 4. Errors and interpretation

What Random Forest leans on, and three concrete cases it gets wrong.

In [4]:
from sklearn.inspection import permutation_importance

gini_importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("RF built-in (Gini) importance:")
print(gini_importance.round(3))

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_SEED, scoring="roc_auc")
perm_importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
print("\npermutation importance (drop in ROC-AUC when shuffled):")
print(perm_importance.round(4))

RF built-in (Gini) importance:
content_age_days                   0.273
ctr_first_half                     0.259
days_active_first_half             0.136
clicks_first_half                  0.113
avg_position_first_half            0.106
impressions_first_half             0.106
ga4_engaged_sessions_first_half    0.008
dtype: float64



permutation importance (drop in ROC-AUC when shuffled):
ctr_first_half                     0.0452
days_active_first_half             0.0234
clicks_first_half                  0.0158
impressions_first_half             0.0141
avg_position_first_half            0.0040
ga4_engaged_sessions_first_half   -0.0002
content_age_days                  -0.0023
dtype: float64


**The Gini-vs-permutation gap IS the finding, not a footnote.** Built-in importance ranks
`content_age_days` #1 (0.273) -- but permutation importance says shuffling it barely moves
ROC-AUC at all (-0.0023, i.e. no real effect, within noise). This is the exact pitfall
scikit-learn's docs warn about: Gini importance is inflated for continuous, high-cardinality
features like a day-count, regardless of whether they actually help. Permutation importance
(the trustworthy one here) instead ranks `ctr_first_half` (0.045) and `days_active_first_half`
(0.023) on top -- and `content_age_days` coming up empty **matches ML-07's own signal check
last week**, which called content age MIXED and explicitly kept it out of the rule. The model
and last week's manual bucket table agree: content age isn't real signal for this lane, twice
confirmed now, two different ways.

In [5]:
test_scored = test.copy()
test_scored["rf_score"] = scores["Random Forest"]
test_scored["rf_pred"] = (test_scored["rf_score"] >= 0.5).astype(int)

fp = test_scored[(test_scored["rf_pred"] == 1) & (test_scored["is_declining_label"] == 0)].sort_values("rf_score", ascending=False)
fn = test_scored[(test_scored["rf_pred"] == 0) & (test_scored["is_declining_label"] == 1)].sort_values("rf_score")

cols = ["impressions_first_half", "clicks_first_half", "clicks_second_half", "ctr_first_half",
        "avg_position_first_half", "content_age_days", "rf_score"]
print(f"false positives: {len(fp)}  |  false negatives: {len(fn)}\n")
print("top false positive (most confident 'declining' call that was wrong):")
print(fp[cols].head(1).to_string(index=False))
print("\ntwo false negatives (missed real declines):")
print(fn[cols].head(3).iloc[[0, 2]].to_string(index=False))

false positives: 5173  |  false negatives: 2192

top false positive (most confident 'declining' call that was wrong):
 impressions_first_half  clicks_first_half  clicks_second_half  ctr_first_half  avg_position_first_half  content_age_days  rf_score
                  486.0                6.0                 7.0        0.012346                  8.94647                39  0.702276

two false negatives (missed real declines):
 impressions_first_half  clicks_first_half  clicks_second_half  ctr_first_half  avg_position_first_half  content_age_days  rf_score
                  311.0                8.0                 3.0        0.025723                 3.140983                 9  0.221597
                  857.0                4.0                 0.0        0.004667                13.945574               276  0.223146


**Three concrete wrong cases:**

1. **False positive -- 486 impr, 6 -> 7 clicks, position 8.9, rf_score 0.70.** The model called
   this "declining" with high confidence; the page actually held (even gained one click).
   At 6 first-half clicks, a swing of a single click flips the binary label -- this is the
   proxy label being noisy at low absolute volume, not the model being wrong in any
   meaningful sense. No feature set fixes a coin-flip label.
2. **False negative -- 311 impr, 8 -> 3 clicks, position 3.1 (top 3!), content only 9 days
   old, rf_score 0.22.** A real 5-click drop, missed with high confidence in the wrong
   direction. Top position and a very fresh page read as "healthy" to the model -- but brand
   -new content often gets an initial curiosity spike that cools within weeks (novelty decay),
   and none of the seven honest features here capture a trend *within* the first half, only
   its total. A week-over-week feature would likely catch this; a single 15-day sum can't.
3. **False negative -- 857 impr, 4 -> 0 clicks, position 13.9, content 276 days old, rf_score
   0.22.** The bigger miss: clicks didn't just fall, they hit zero, and the model was
   confident it wouldn't. Older, lower-position, already-modest-CTR pages like this one are
   exactly where a small further decline is easy to miss because nothing in the first-half
   window signals it's about to go to zero rather than staying flat.

**Honest summary:** Random Forest clearly beats the frozen baseline and both Logistic
Regression variants at every K tested (precision@50: 0.78 vs 0.58 best alternative), and the
gain traces to two real signals (`ctr_first_half`, `days_active_first_half`) confirmed by
permutation importance, not to a feature that only looks important. The errors that remain are
mostly proxy-label noise at low click volumes and missing within-window trend features --
both are capstone-scope fixes, not reasons to distrust today's comparison.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.